# BabyLM 100M original + cloned-language GPT

This notebook runs the complete BabyLM experiment in Google Colab: official data preparation, SentencePiece BPE training, tokenization, checkpointed pretraining, training/PPL plots, and matched original/clone BLiMP SVA evaluation. BabyLM data is downloaded only inside Colab; reusable token assets and experiment outputs are saved to Google Drive.

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import shutil
import subprocess

REPO_URL = "https://github.com/jiayi-ji01/mllms-colab.git"
PROJECT_DIR = Path("/content/mllms-colab")
DRIVE_ROOT = Path("/content/drive/MyDrive/mllms-colab")
RUN_DIR = DRIVE_ROOT / "runs/gpt12_babylm_clone_colab"
ASSET_DIR = DRIVE_ROOT / "assets/babylm_100m_bpe16k"
CONFIG_PATH = PROJECT_DIR / "configs/gpt12_babylm_clone_colab.yaml"
TOKENIZER_PATH = PROJECT_DIR / "artifacts/babylm_tokenizer/tokenizer.model"
PROCESSED_DIR = PROJECT_DIR / "data/babylm/processed"

RUN_TRAINING = True
RUN_SVA_LOGIT = True
RUN_CONDITIONAL_COMPARISON = True

## 1. GPU and Google Drive

In [ ]:
import torch
from google.colab import drive

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print("GPU:", torch.cuda.get_device_name(0))
print("Precision:", "bf16" if torch.cuda.is_bf16_supported() else "fp16")
drive.mount("/content/drive")

## 2. Clone/update and install

In [ ]:
if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
print("Project:", PROJECT_DIR)

## 3. Prepare or restore BabyLM assets

On the first run this downloads the official cleaned BabyLM 100M-word train corpus plus official dev/test inside Colab. Later sessions restore the tokenizer and binary token streams from Drive without downloading BabyLM again.

In [ ]:
local_assets = {
    "tokenizer.model": TOKENIZER_PATH,
    "tokenizer.vocab": TOKENIZER_PATH.with_suffix(".vocab"),
    "train.bin": PROCESSED_DIR / "train.bin",
    "validation.bin": PROCESSED_DIR / "validation.bin",
    "test.bin": PROCESSED_DIR / "test.bin",
    "token_counts.json": PROCESSED_DIR / "token_counts.json",
}
local_ready = all(path.is_file() for path in local_assets.values())
backup_ready = all((ASSET_DIR / name).is_file() for name in local_assets)

if not local_ready and backup_ready:
    for name, destination in local_assets.items():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ASSET_DIR / name, destination)
    local_ready = True
    print("Restored BabyLM tokenizer and token streams from Drive.")

if not local_ready:
    subprocess.run(["mllms", "data", "prepare-babylm", "--output-dir", "data/babylm/raw"], check=True)
    subprocess.run([
        "mllms", "tokenizer", "train",
        "--input", "data/babylm/raw/train.txt",
        "--model-prefix", "artifacts/babylm_tokenizer/tokenizer",
        "--vocab-size", "16000",
        "--input-sentence-size", "5000000",
    ], check=True)
    subprocess.run([
        "mllms", "data", "tokenize",
        "--input-dir", "data/babylm/raw",
        "--output-dir", str(PROCESSED_DIR),
        "--tokenizer", str(TOKENIZER_PATH),
        "--no-train-token-limit",
    ], check=True)
    ASSET_DIR.mkdir(parents=True, exist_ok=True)
    for name, source in local_assets.items():
        shutil.copy2(source, ASSET_DIR / name)
    raw_manifest = PROJECT_DIR / "data/babylm/raw/manifest.json"
    if raw_manifest.is_file():
        shutil.copy2(raw_manifest, ASSET_DIR / "raw_manifest.json")
    print("Prepared BabyLM assets and backed them up to Drive.")

token_counts = json.loads((PROCESSED_DIR / "token_counts.json").read_text())
print(json.dumps(token_counts, indent=2))
for name, path in local_assets.items():
    print(f"{name}: {path.stat().st_size / 2**20:.1f} MiB")

## 4. Resolved training size

In [ ]:
train_tokens = int(token_counts["splits"]["train"]["tokens"])
tokens_per_step = 4 * 8 * 256
optimizer_steps = math.ceil(2.0 * train_tokens / tokens_per_step)
planned_tokens = optimizer_steps * tokens_per_step

print(f"Actual encoded train tokens: {train_tokens:,}")
print(f"Tokens per optimizer step: {tokens_per_step:,}")
print(f"Optimizer steps: {optimizer_steps:,}")
print(f"Planned seen tokens: {planned_tokens:,}")
print(f"Nominal epochs: {planned_tokens / train_tokens:.4f}")
print("Model parameters: 54,344,704")

## 5. Train or resume

The newest valid `latest.pt` or `best.pt` is selected by checkpoint step. `final.pt` skips training. All checkpoints are stored in the dedicated BabyLM Drive directory.

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
final_checkpoint = RUN_DIR / "final.pt"
latest_checkpoint = RUN_DIR / "latest.pt"
best_checkpoint = RUN_DIR / "best.pt"

def checkpoint_step(path):
    if not path.is_file():
        return -1
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        step = int(checkpoint["step"])
        del checkpoint
        gc.collect()
        return step
    except Exception as error:
        print(f"Ignoring invalid checkpoint {path.name}: {error}")
        return -1

if RUN_TRAINING and not final_checkpoint.is_file():
    command = [
        "mllms", "train",
        "--config", str(CONFIG_PATH),
        "--output-dir", str(RUN_DIR),
    ]
    candidates = [latest_checkpoint, best_checkpoint]
    resume_checkpoint = max(candidates, key=checkpoint_step)
    resume_step = checkpoint_step(resume_checkpoint)
    if resume_step >= 0:
        command.extend(["--resume", str(resume_checkpoint)])
        print(f"Resuming from {resume_checkpoint.name}, step {resume_step:,}")
    subprocess.run(command, check=True)
elif final_checkpoint.is_file():
    print("Training already complete:", final_checkpoint)
else:
    print("Training skipped because RUN_TRAINING=False")

assert best_checkpoint.is_file(), "best.pt is required for evaluation"
print("Best checkpoint step:", checkpoint_step(best_checkpoint))

## 6. Training loss and perplexity

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

subprocess.run(["mllms", "plot", "training", "--run-dir", str(RUN_DIR)], check=True)
display(Image(filename=str(RUN_DIR / "training_report.png")))
display(pd.read_csv(RUN_DIR / "training_summary.csv"))

records = [json.loads(line) for line in (RUN_DIR / "train_log.jsonl").read_text().splitlines() if line.strip()]
train = pd.DataFrame(row for row in records if row.get("type") == "train").drop_duplicates("step", keep="last").sort_values("step")
validation = pd.DataFrame(row for row in records if row.get("type") == "validation").drop_duplicates("step", keep="last").sort_values("step")
train["ppl"] = np.exp(train["train_loss"].clip(upper=20))
train["smoothed_ppl"] = train["ppl"].rolling(50, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train["step"], train["smoothed_ppl"], label="Train PPL, smoothed")
ax.plot(validation["step"], validation["original_perplexity"], marker="o", label="Validation original")
ax.plot(validation["step"], validation["clone_perplexity"], marker="o", label="Validation clone")
ax.set_yscale("log")
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Perplexity, log scale")
ax.set_title("BabyLM pretraining perplexity")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
ppl_report = RUN_DIR / "perplexity_report.png"
fig.savefig(ppl_report, dpi=180)
plt.show()

## 7. Prepare BLiMP SVA pairs

BLiMP is evaluation-only and is never mixed into pretraining.

In [ ]:
blimp_raw = PROJECT_DIR / "data/blimp/raw/agreement.jsonl"
blimp_processed = PROJECT_DIR / "data/blimp/processed/babylm_agreement.jsonl"
if not blimp_raw.is_file():
    subprocess.run(["mllms", "blimp", "download"], check=True)
if not blimp_processed.is_file():
    subprocess.run([
        "mllms", "blimp", "prepare",
        "--checkpoint", str(best_checkpoint),
        "--tokenizer", str(TOKENIZER_PATH),
        "--output", str(blimp_processed),
    ], check=True)
print("Prepared BLiMP data:", blimp_processed)

## 8. Strict verb-logit SVA evaluation

This uses identical original/clone examples and computes `logit(correct_verb) - logit(incorrect_verb)`. Only identical-prefix examples with one-token alternatives are included; exclusions are reported.

In [ ]:
def result_is_current(summary_path, expected_step, expected_scoring):
    if not summary_path.is_file():
        return False
    try:
        summary = json.loads(summary_path.read_text())
        return summary.get("checkpoint_step") == expected_step and summary.get("scoring") == expected_scoring
    except Exception:
        return False

best_step = checkpoint_step(best_checkpoint)
sva_logit_dir = RUN_DIR / "blimp_sva_logit"
sva_logit_summary = sva_logit_dir / "agreement_summary.json"
if RUN_SVA_LOGIT and not result_is_current(sva_logit_summary, best_step, "verb-logit"):
    subprocess.run([
        "mllms", "blimp", "evaluate",
        "--checkpoint", str(best_checkpoint),
        "--tokenizer", str(TOKENIZER_PATH),
        "--data", str(blimp_processed),
        "--scoring", "verb-logit",
        "--device", "cuda",
        "--output-dir", str(sva_logit_dir),
    ], check=True)

if sva_logit_summary.is_file():
    subprocess.run(["mllms", "plot", "blimp", "--results-dir", str(sva_logit_dir)], check=True)
    display(Image(filename=str(sva_logit_dir / "blimp_report.png")))
    display(pd.read_csv(sva_logit_dir / "blimp_summary.csv"))
    print(json.dumps(json.loads(sva_logit_summary.read_text())["overall"], indent=2))

## 9. Conditional-logprob comparison with the TinyStories experiment

In [ ]:
conditional_dir = RUN_DIR / "blimp_conditional"
conditional_summary = conditional_dir / "agreement_summary.json"
if RUN_CONDITIONAL_COMPARISON and not result_is_current(conditional_summary, best_step, "conditional-logprob"):
    subprocess.run([
        "mllms", "blimp", "evaluate",
        "--checkpoint", str(best_checkpoint),
        "--tokenizer", str(TOKENIZER_PATH),
        "--data", str(blimp_processed),
        "--scoring", "conditional-logprob",
        "--device", "cuda",
        "--output-dir", str(conditional_dir),
    ], check=True)

if conditional_summary.is_file():
    subprocess.run(["mllms", "plot", "blimp", "--results-dir", str(conditional_dir)], check=True)
    display(Image(filename=str(conditional_dir / "blimp_report.png")))
    display(pd.read_csv(conditional_dir / "blimp_summary.csv"))

## 10. Final result inventory

In [ ]:
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(RUN_DIR)}  ({path.stat().st_size / 2**20:.2f} MiB)")